# Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
accounts = spark.table("aml_framework.accounts")
alerts = spark.table("aml_framework.alerts")
transactions = spark.table("aml_framework.transactions")
saml_d = spark.table("aml_framework.saml_d")

# Transaction Count

In [0]:
txn_ct = transactions.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    F.count("*").alias("num_transactions")
)
txn_ct.display()

# Unique beneficiaries

In [0]:
unique_beneficiaries = transactions.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    F.countDistinct("RECEIVER_ACCOUNT_ID")
    .alias("unique_receivers")
)
unique_beneficiaries.display()

# Outgoing Transaction Volume

In [0]:
amount_sent = transactions.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    F.sum("TX_AMOUNT").alias("total_amt_sent")
)
amount_sent.display()

# Incoming Transaction Volume

In [0]:
amount_received = transactions.groupBy(
    "RECEIVER_ACCOUNT_ID"
).agg(
    F.sum("TX_AMOUNT").alias("total_amt_received")
)
amount_received.display()

# Average Outgoing Transactions

In [0]:
avg_txn_out = transactions.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    F.avg("TX_AMOUNT").alias("avg_txn_amt_out")
)
avg_txn_out.display()

# Average Incoming Transactions

In [0]:
avg_txn_in = transactions.groupBy(
    "RECEIVER_ACCOUNT_ID"
).agg(
    F.avg("TX_AMOUNT").alias("avg_txn_amt_in")
)
avg_txn_in.display()

# Max Min Transactions

In [0]:
max_min_txn = transactions.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    F.max("TX_AMOUNT").alias("max_txn_amt"),
    F.min("TX_AMOUNT").alias("min_txn_amt")
)
max_min_txn.display()

# Number of Alerts

In [0]:
alert_features = alerts.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    F.count("*").alias("alert_count")
)
alert_features.display()

# Number of Fraud alerts

In [0]:
fraud_alerts = alerts.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    F.sum(F.col("IS_FRAUD").cast("int"))
    .alias("fraud_alert_count")
)
fraud_alerts.display()

# Transaction velocity

In [0]:
txn_velocity = transactions.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    (
        F.count("*") /
        (F.max("TIMESTAMP") - F.min("TIMESTAMP") + 1)
    ).alias("txn_per_step")
)
txn_velocity.display()

# Activity Span

In [0]:
activity_span = transactions.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    (
        F.max("TIMESTAMP") -
        F.min("TIMESTAMP")
    ).alias("active_span")
)
activity_span.display()

# Recent Transactions

In [0]:
latest_step = transactions.agg(
    F.max("TIMESTAMP")
).collect()[0][0]

recent_txns = transactions.filter(
    F.col("TIMESTAMP") >= latest_step - 20
)

recent_txn_count = recent_txns.groupBy(
    "SENDER_ACCOUNT_ID"
).count().withColumnRenamed("count", "number_of_recent_trxns")

recent_txn_count.display()
recent_txns.display()

# Max transactions in one step

In [0]:
burstiness = transactions.groupBy(
    "SENDER_ACCOUNT_ID",
    "TIMESTAMP"
).count()

burst_feature = burstiness.groupBy(
    "SENDER_ACCOUNT_ID"
).agg(
    F.max("count").alias("max_txns_in_one_step")
)

burst_feature.display()

# Non beneficiary or Non Sender

In [0]:
senders = transactions.select("SENDER_ACCOUNT_ID").distinct()
receivers = transactions.select("RECEIVER_ACCOUNT_ID").distinct()
all_accounts = accounts.select("ACCOUNT_ID").distinct()

non_beneficiary_accts = all_accounts.join(receivers.withColumnRenamed("RECEIVER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "leftanti")
non_sender_accts = all_accounts.join(senders.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "leftanti")

non_beneficiary_accts.display()
non_sender_accts.display()

# Number of International Transactions

In [0]:
# Sender account -> country
sender_accounts = (
    accounts
    .select(
        F.col("ACCOUNT_ID").alias("SENDER_ACCOUNT_ID"),
        F.col("COUNTRY").alias("SENDER_COUNTRY")
    )
)

# Receiver account -> country
receiver_accounts = (
    accounts
    .select(
        F.col("ACCOUNT_ID").alias("RECEIVER_ACCOUNT_ID"),
        F.col("COUNTRY").alias("RECEIVER_COUNTRY")
    )
)

# Attach countries to transactions
txn_with_country = (
    transactions
    .join(sender_accounts, "SENDER_ACCOUNT_ID", "left")
    .join(receiver_accounts, "RECEIVER_ACCOUNT_ID", "left")
)

# International transactions
international_txns = txn_with_country.filter(
    F.col("SENDER_COUNTRY") != F.col("RECEIVER_COUNTRY")
)

# Count
international_count = international_txns.count()

print(f"International Transactions: {international_count}")
international_txns.display()

# Combine all

In [0]:
processed_table = accounts.join(txn_ct.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(unique_beneficiaries.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(amount_sent.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(amount_received.withColumnRenamed("RECEIVER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .withColumn("out_vs_in_amt_ratio", F.col("total_amt_sent")/F.col("total_amt_received")) \
    .join(avg_txn_out.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(avg_txn_in.withColumnRenamed("RECEIVER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(max_min_txn.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(alert_features.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(fraud_alerts.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(txn_velocity.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(recent_txn_count.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(activity_span.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(burst_feature.withColumnRenamed("SENDER_ACCOUNT_ID", "ACCOUNT_ID"), "ACCOUNT_ID", "left") \
    .join(non_beneficiary_accts, "ACCOUNT_ID", "left") \
    .join(non_sender_accts, "ACCOUNT_ID", "left") \
    .fillna(0, subset=['num_transactions','unique_receivers','total_amt_sent','total_amt_received','out_vs_in_amt_ratio','avg_txn_amt_out','avg_txn_amt_in','max_txn_amt','min_txn_amt','alert_count','fraud_alert_count','txn_per_step','number_of_recent_trxns', 'active_span','max_txns_in_one_step'])
processed_table.display()

In [0]:
processed_table.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("aml_framework.aml_preprocessed_table")

In [0]:
processed_table.columns